# Agri Vision — three-client GPU training

Select **Runtime → Change runtime type → GPU**, edit the settings below if needed, then **Runtime → Run all** and approve Drive access.

Upload `original_dataset_colab.zip` and `agrivision_colab_code.zip` from your project folder to `MyDrive/Agri Vision/` first. Keep your existing Drive files. The code bundle supplies the corrected 30-class YAML, never the old 23-class YAML inside the dataset ZIP. It runs three clients sequentially on one GPU and performs sample-weighted FedAvg after all three finish. This is a local simulation; no distributed workers or exposed server are started.

GPU availability and session duration are limited. Only completed global rounds can resume; interruption mid-round requires repeating that round. The class names currently in GitHub were reconstructed; passing numeric validation does not independently verify annotation semantics. Verify names against your original dataset export before reporting disease-specific accuracy.

In [ ]:
from pathlib import Path
ROUNDS = 10
LOCAL_EPOCHS = 1
BATCH = 4
IMGSZ = 320
DATASET_ZIP = '/content/drive/MyDrive/Agri Vision/original_dataset_colab.zip'
CODE_ZIP = '/content/drive/MyDrive/Agri Vision/agrivision_colab_code.zip'
OUTPUT_ROOT = '/content/drive/MyDrive/Agri Vision/colab_results'
# Optional: full Drive path to a completed global_round_XXX.pt
INITIAL_CHECKPOINT = ''


In [ ]:
from google.colab import drive
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU, then Run all again.')
print('GPU:', torch.cuda.get_device_name(0))
drive.mount('/content/drive')
if not Path(DATASET_ZIP).is_file():
    raise FileNotFoundError(f'Upload original_dataset_colab.zip here first: {DATASET_ZIP}')
if not Path(CODE_ZIP).is_file():
    raise FileNotFoundError(f'Upload agrivision_colab_code.zip here first: {CODE_ZIP}')


## Load the corrected project code

The small Drive bundle avoids GitHub authentication and excludes the Windows virtual environment. Its SHA-256 hash is recorded so the experiment can be reproduced.

In [ ]:
import hashlib, uuid, os, subprocess, sys, zipfile
WORK = Path('/content') / ('agrivision_' + uuid.uuid4().hex[:8])
WORK.mkdir()
CODE_BUNDLE_SHA256 = hashlib.sha256(Path(CODE_ZIP).read_bytes()).hexdigest()
with zipfile.ZipFile(CODE_ZIP) as bundle:
    bundle.extractall(WORK)
os.chdir(WORK)
print('Code bundle SHA-256:', CODE_BUNDLE_SHA256)
print('Working directory:', WORK)
# Keep Colab GPU PyTorch; do not copy the Windows environment.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(WORK / 'requirements.txt')], check=True, cwd=WORK)


## Prepare data on Colab's local disk

The ZIP must contain `original_dataset/train`, `valid`, and `test` with `images` and `labels`. The notebook extracts only images and label text, preserves the current GitHub YAML, and rebuilds the three disjoint shards with seed 42.

In [ ]:
import shutil, zipfile
archive = WORK / 'source.zip'
shutil.copy2(DATASET_ZIP, archive)
with zipfile.ZipFile(archive) as z:
    for entry in z.infolist():
        parts = Path(entry.filename).parts
        if entry.is_dir():
            continue
        if len(parts) != 4 or parts[0] != 'original_dataset' or parts[1] not in ('train', 'valid', 'test') or parts[2] not in ('images', 'labels'):
            continue
        suffix = Path(parts[3]).suffix.lower()
        if (parts[2] == 'labels' and suffix != '.txt') or (parts[2] == 'images' and suffix not in ('.jpg', '.jpeg', '.png', '.bmp', '.webp')):
            continue
        target = WORK.joinpath(*parts)
        target.parent.mkdir(parents=True, exist_ok=True)
        with z.open(entry) as source, target.open('wb') as dest:
            shutil.copyfileobj(source, dest)
subprocess.run([sys.executable, '-u', str(WORK / 'split_dataset.py'), '--rebuild', '--seed', '42'], check=True, cwd=WORK)


## Train and save to Drive

Each round trains client1, client2, then client3 from the same starting global model. A global checkpoint is saved immediately after aggregation, followed by validation metrics. Live output is shown below and copied to Drive. Stop the cell to stop its training process. Select a new run to restart; never overwrite an earlier run.

In [ ]:
from datetime import datetime, timezone
RUN = Path(OUTPUT_ROOT) / (datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6])
RUN.parent.mkdir(parents=True, exist_ok=True)
command = [sys.executable, '-u', str(WORK / 'colab_train.py'), '--rounds', str(ROUNDS), '--local-epochs', str(LOCAL_EPOCHS), '--batch', str(BATCH), '--imgsz', str(IMGSZ), '--output', str(RUN)]
if INITIAL_CHECKPOINT:
    if not Path(INITIAL_CHECKPOINT).is_file():
        raise FileNotFoundError(INITIAL_CHECKPOINT)
    command += ['--initial-checkpoint', INITIAL_CHECKPOINT]
log_path = RUN.parent / (RUN.name + '.log')
(RUN.parent / (RUN.name + '_code_bundle_sha256.txt')).write_text(CODE_BUNDLE_SHA256)
print('Results:', RUN)
print('Log:', log_path)
with log_path.open('w') as log:
    process = subprocess.Popen(command, cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        result = process.wait()
        if result:
            raise RuntimeError(f'Training exited with code {result}; inspect the log above.')
    finally:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
print('Saved checkpoints:', list(RUN.glob('global_round_*.pt')))


## Next session

Use the newest completed `global_round_XXX.pt` as `INITIAL_CHECKPOINT`, choose how many **additional** rounds to train, then Run all. The dataset remains on Drive; local copies disappear when the runtime is recycled. Old 23-class checkpoints are incompatible. Validation metrics are reported each round; keep the test split for the final evaluation. One successful run confirms execution, not model accuracy.